# 06 - Demo Cleanup: Teardown All Resources

This notebook destroys all Snowflake objects created by this demo in **reverse creation order**.
Run it after a demo to restore the account to a clean state so the demo can be repeated.

> **Warning**: This is destructive and irreversible. All models, services, data, and infrastructure will be permanently deleted.

## Teardown Order (reverse of creation)

| Step | Object(s) Removed | Type |
|------|-------------------|------|
| ← 4 | `PATIENT_RISK_MONITOR`, `MONITOR_BASELINE`, `INFERENCE_LOGS_VIEW` | Monitor, Table, View |
| ← 3 | `PATIENT_RISK_SERVICE` | SPCS Service |
| ← 2 | `PATIENT_RISK_MODEL` (all versions) | Model Registry |
| ← 1 | `PATIENT_FEATURES` feature view, `PATIENT` entity, `TRAINING_FEATURES`, `TEST_FEATURES` | Feature Store, Tables |
| ← DAG | Tasks, Stored Procedures, `PIPELINE_EXECUTIONS` | Tasks, Procedures, Table |
| ← Data | `RAW_PATIENT_DATA`, `STREAMING_PATIENT_DATA` (truncated) | Table data |
| ← Setup | All remaining tables, stages | Tables, Stages |
| ← Setup | `PYPI_ACCESS_INTEGRATION`, `PYPI_NETWORK_RULE` | Network objects |
| ← Setup | `ML_DEMO_COMPUTE_POOL` | Compute Pool |
| ← Setup | `ML_DEMO_PIPELINE_DB`, `ML_DEMO_WAREHOUSE` | Database, Warehouse |

## Imports and Configuration

In [ ]:
%cd ..

In [ ]:
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

from source.configs import get_config
from source.utils import get_session

config = get_config("source/config.yaml")

DB     = config.snowflake.database
SCHEMA = config.snowflake.schema_name
WH     = config.snowflake.warehouse
POOL   = config.compute.compute_pool
MODEL  = config.model.model_name
FQS    = f"{DB}.{SCHEMA}"

print(f"Database : {DB}")
print(f"Schema   : {SCHEMA}")
print(f"Warehouse: {WH}")
print(f"Pool     : {POOL}")
print(f"Model    : {MODEL}")

In [ ]:
session = get_session("DEMO")
session.use_database(DB)
session.use_schema(SCHEMA)
session.use_warehouse(WH)

print(f"Connected as : {session.get_current_user()}")
print(f"Current role : {session.get_current_role()}")

---
## ← Step 4: Drop Model Monitor

Remove the monitor, its baseline snapshot table, and the inference logs view.

In [ ]:
from source.framework.monitor import ModelMonitor

MONITOR_NAME   = "PATIENT_RISK_MONITOR"
BASELINE_TABLE = f"{FQS}.MONITOR_BASELINE"
LOGS_VIEW      = f"{FQS}.INFERENCE_LOGS_VIEW"

monitor = ModelMonitor(session=session, database=DB, schema=SCHEMA)
monitor.drop_monitor(MONITOR_NAME)
print(f"Monitor '{MONITOR_NAME}' dropped")

session.sql(f"DROP TABLE IF EXISTS {BASELINE_TABLE}").collect()
print(f"Table 'MONITOR_BASELINE' dropped")

session.sql(f"DROP VIEW IF EXISTS {LOGS_VIEW}").collect()
print(f"View 'INFERENCE_LOGS_VIEW' dropped")

---
## ← Step 3: Drop REST Inference Service

Stop and remove the SPCS service.

In [ ]:
from source.framework.deploy import ModelDeployer

SERVICE_NAME = "PATIENT_RISK_SERVICE"

deployer = ModelDeployer(
    session=session,
    registry_database=DB,
    registry_schema=SCHEMA,
)

status = deployer.get_service_status(SERVICE_NAME)
print(f"Service '{SERVICE_NAME}' current status: {status}")

deployer.drop_service(SERVICE_NAME)
print(f"Service '{SERVICE_NAME}' dropped")

---
## ← Step 2: Drop Model from Registry

Delete all versions of the model from the Snowflake Model Registry.

In [ ]:
from snowflake.ml.registry import Registry

registry = Registry(session, database_name=DB, schema_name=SCHEMA)

try:
    model = registry.get_model(MODEL)
    versions = model.versions()
    print(f"Found {len(versions)} version(s) for model '{MODEL}':")
    for v in versions:
        print(f"  - {v.version_name}")
    registry.delete_model(MODEL)
    print(f"Model '{MODEL}' and all versions deleted")
except Exception as e:
    print(f"Model '{MODEL}' not found or already deleted: {e}")

---
## ← Step 1: Drop Feature Store Objects

Suspend and delete the feature view, delete the entity, then drop the feature tables.

In [ ]:
from snowflake.ml.feature_store import FeatureStore, CreationMode

fs = FeatureStore(
    session=session,
    database=DB,
    name=SCHEMA,
    default_warehouse=WH,
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST,
)

try:
    fv = fs.get_feature_view("PATIENT_FEATURES", "v1")
    fs.suspend_feature_view(fv)
    print("Feature view 'PATIENT_FEATURES v1' suspended")
    fs.delete_feature_view(fv)
    print("Feature view 'PATIENT_FEATURES v1' deleted")
except Exception as e:
    print(f"Feature view not found or already deleted: {e}")

try:
    fs.delete_entity("PATIENT")
    print("Entity 'PATIENT' deleted")
except Exception as e:
    print(f"Entity not found or already deleted: {e}")

In [ ]:
for table in ["TRAINING_FEATURES", "TEST_FEATURES"]:
    session.sql(f"DROP TABLE IF EXISTS {FQS}.{table}").collect()
    print(f"Table '{table}' dropped")

---
## ← DAG: Drop Tasks and Stored Procedures

Suspend the root task, drop all tasks and stored procedures, then drop the execution log table.

In [ ]:
try:
    session.sql(f"ALTER TASK IF EXISTS {FQS}.PIPELINE_ROOT_TASK SUSPEND").collect()
    print("Root task suspended")
except Exception as e:
    print(f"Could not suspend root task (may not exist): {e}")

In [ ]:
from source.pipeline.dag import teardown_dag

teardown_dag(session, config)
print("All tasks and stored procedures dropped")

---
## ← Data: Clear Patient Data Tables

Truncate the raw and streaming patient data tables before dropping them with the rest of setup.

In [ ]:
for table in ["STREAMING_PATIENT_DATA", "RAW_PATIENT_DATA"]:
    try:
        session.sql(f"TRUNCATE TABLE IF EXISTS {FQS}.{table}").collect()
        print(f"Table '{table}' truncated")
    except Exception as e:
        print(f"Could not truncate '{table}': {e}")

---
## ← Setup: Drop Remaining Tables

Drop all tables created during infrastructure setup (in reverse creation order).

In [ ]:
tables = [
    "STREAMING_FEATURES",
    "TEST_FEATURES",
    "BASELINE_PATIENT_DATA",
    "MODEL_METRICS",
    "STREAMING_PATIENT_DATA",
    "RAW_PATIENT_DATA",
    "PIPELINE_EXECUTIONS", 
    "PIPELINE_STATE"
]

for table in tables:
    session.sql(f"DROP TABLE IF EXISTS {FQS}.{table}").collect()
    print(f"Table '{table}' dropped")

---
## ← Setup: Drop Stages

Drop internal stages for model artifacts, job payloads, and data (in reverse creation order).

In [ ]:
stages = [
    "EVALUATION_RESULTS",
    "DATA_STAGE",
    "JOB_PAYLOADS",
    "MODEL_ARTIFACTS",
]

for stage in stages:
    session.sql(f"DROP STAGE IF EXISTS {FQS}.{stage}").collect()
    print(f"Stage '{stage}' dropped")

---
## ← Setup: Drop Network Rule and External Access Integration

Remove the PyPI egress network rule and its external access integration.
Dropping the integration requires ACCOUNTADMIN.

In [ ]:
INTEGRATION  = "PYPI_ACCESS_INTEGRATION"
NETWORK_RULE = f"{FQS}.PYPI_NETWORK_RULE"
original_role = session.get_current_role()

try:
    session.sql("USE ROLE ACCOUNTADMIN").collect()
    session.sql(f"DROP EXTERNAL ACCESS INTEGRATION IF EXISTS {INTEGRATION}").collect()
    print(f"External access integration '{INTEGRATION}' dropped")
except Exception as e:
    print(f"Could not drop integration (may require ACCOUNTADMIN): {e}")
finally:
    session.sql(f"USE ROLE {original_role}").collect()

session.sql(f"DROP NETWORK RULE IF EXISTS {NETWORK_RULE}").collect()
print(f"Network rule 'PYPI_NETWORK_RULE' dropped")

---
## ← Setup: Suspend and Drop Compute Pool

Suspend the compute pool to stop billing, then drop it.

In [ ]:
try:
    session.sql(f"ALTER COMPUTE POOL IF EXISTS {POOL} STOP ALL").collect()
    print(f"Compute pool '{POOL}' — all services stopped")
except Exception as e:
    print(f"Could not stop services on pool: {e}")

try:
    session.sql(f"ALTER COMPUTE POOL IF EXISTS {POOL} SUSPEND").collect()
    print(f"Compute pool '{POOL}' suspended")
except Exception as e:
    print(f"Could not suspend pool: {e}")

session.sql(f"DROP COMPUTE POOL IF EXISTS {POOL}").collect()
print(f"Compute pool '{POOL}' dropped")

---
## ← Setup: Drop Database and Warehouse

Drop the database (cascades to schema, all remaining tables, views, models, and feature store objects)
then drop the warehouse.

In [ ]:
session.sql(f"DROP DATABASE IF EXISTS {DB}").collect()
print(f"Database '{DB}' dropped (schema, tables, models, feature store all removed)")

In [ ]:
session.sql(f"DROP WAREHOUSE IF EXISTS {WH}").collect()
print(f"Warehouse '{WH}' dropped")

---
## Cleanup Complete

All demo resources have been removed. The account is back to a clean state.

To re-run the demo, follow the **Pre-Demo Checklist** in `docs/DEMO_GUIDE.md`:

```bash
python -m setup.database_setup
python -m setup.stages_setup
python -m setup.compute_pool_setup
python -m setup.tables_setup
python -m data.historical
python -m source.pipeline.dag
```